# BigGAN + MidJourney: Cat 4 Classes, IQA + FFT + XGBoost / MLP

Notebook ini menggabungkan dua keluarga fitur: **NR-IQA** (`PIQE`, `BRISQUE`, `NIQE`) dan **FFT mean**. Setup data tetap sama seperti notebook `08`?`11`.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import pyiqa
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from transformers import CLIPModel, CLIPProcessor
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..")
MULTICLASS_ROOT = PROJECT_ROOT / "data" / "raw" / "genimage_multiclass_balanced"
DATA_ROOT = MULTICLASS_ROOT / "genimage"
DATA_ROOTS = {
    "BigGAN": DATA_ROOT / "BigGAN" / "train",
    "MidJourney": DATA_ROOT / "MidJourney" / "train",
}
MANIFEST_SOURCE = MULTICLASS_ROOT / "genimage_multiclass_manifest.csv"
CLASS_TABLE_SOURCE = MULTICLASS_ROOT / "genimage_multiclass_classes.csv"
IMAGENET_MAP = PROJECT_ROOT / "data" / "imagenet_class_index.json"
print("Data root:", DATA_ROOT.resolve())
print("Manifest source:", MANIFEST_SOURCE.resolve())
SUBSET_NAME = "biggan_midjourney_multiclass_train"
MANIFEST_OUT = PROJECT_ROOT / "data" / "genimage_manifest_biggan_midjourney_multiclass_train.csv"
FFT_OUT = PROJECT_ROOT / "artifacts" / "features_fft_mean_biggan_midjourney_multiclass_train.csv"
IQA_OUT = PROJECT_ROOT / "artifacts" / "features_iqa_biggan_midjourney_multiclass_train.csv"
VECTOR_OUT = PROJECT_ROOT / "artifacts" / "feature_vector_biggan_midjourney_multiclass_iqa_fft.csv"
METRICS_OUT = PROJECT_ROOT / "artifacts" / "results_classification_biggan_midjourney_multiclass_iqa_fft.csv"
PRED_OUT = PROJECT_ROOT / "artifacts" / "predictions_classification_biggan_midjourney_multiclass_iqa_fft.csv"


## 1. Audit data per generator

In [ ]:
manifest_source = pd.read_csv(MANIFEST_SOURCE)
class_table = pd.read_csv(CLASS_TABLE_SOURCE)

manifest_source["path_exists"] = manifest_source["path"].map(lambda p: Path(p).is_file())
print("Manifest source:", MANIFEST_SOURCE.resolve())
print("Class table:", CLASS_TABLE_SOURCE.resolve())
print("Manifest shape:", manifest_source.shape)
print("Jumlah kelas:", manifest_source["imagenet_id"].nunique())
print("Semua path ada:", bool(manifest_source["path_exists"].all()))

display(class_table.head(10))
display(
    manifest_source
    .groupby(["generator", "class_name"])
    .size()
    .rename("count")
    .reset_index()
)
display(
    manifest_source
    .groupby(["generator", "class_name", "imagenet_id", "wordnet_id", "content_label"])
    .size()
    .rename("count")
    .reset_index()
    .describe(include="all")
)


## 2. Bangun manifest gabungan untuk 4 class kucing yang sama


In [ ]:
manifest = pd.read_csv(MANIFEST_SOURCE).copy()
manifest["subset_name"] = "biggan_midjourney_multiclass_train"
manifest["split"] = "train"
manifest["path"] = manifest["path"].map(lambda p: str(Path(p).resolve()))

missing_paths = manifest.loc[~manifest["path"].map(lambda p: Path(p).is_file()), "path"].head()
if len(missing_paths):
    raise FileNotFoundError(
        "Ada path manifest yang tidak ditemukan, contoh: "
        + str(missing_paths.tolist())
    )

manifest.to_csv(MANIFEST_OUT, index=False)
print("Saved manifest:", MANIFEST_OUT.resolve())
print("Manifest shape:", manifest.shape)
display(manifest.head())
display(manifest.groupby(["generator", "y_ai"]).size().rename("count").reset_index())
display(
    manifest
    .groupby(["generator", "class_name", "content_label"])
    .size()
    .rename("count")
    .reset_index()
    .head(20)
)


## 3. Ekstraksi fitur

In [ ]:
if IQA_OUT.exists():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Loaded existing IQA features later; skip IQA metric initialization.")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    metric_piqe = pyiqa.create_metric("piqe", device=device)
    metric_brisque = pyiqa.create_metric("brisque", device=device)
    metric_niqe = pyiqa.create_metric("niqe", device=device)
    print("device:", device)


In [ ]:
if IQA_OUT.exists():
    iqa_df = pd.read_csv(IQA_OUT)
    print("Loaded existing IQA features:", IQA_OUT.resolve())
else:
    from PIL import Image

    IQA_CACHE_DIR = IQA_OUT.parent / "_iqa_safe_images_multiclass"
    IQA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    MIN_IQA_SIDE = 256

    def make_iqa_safe_image(path: str) -> str:
        """NIQE needs enough spatial support; upsample small images to avoid zero-size blocks."""
        src = Path(path)
        img = Image.open(src).convert("RGB")
        w, h = img.size

        if min(w, h) >= MIN_IQA_SIDE:
            return str(src)

        scale = MIN_IQA_SIDE / min(w, h)
        new_size = (int(round(w * scale)), int(round(h * scale)))
        safe_path = IQA_CACHE_DIR / f"{src.stem}_{new_size[0]}x{new_size[1]}.png"

        if not safe_path.exists():
            img = img.resize(new_size, Image.Resampling.BICUBIC)
            img.save(safe_path)

        return str(safe_path)

    def extract_iqa_scores(path: str):
        safe_path = make_iqa_safe_image(path)
        with torch.no_grad():
            piqe = float(metric_piqe(safe_path).item())
            brisque = float(metric_brisque(safe_path).item())
            niqe = float(metric_niqe(safe_path).item())
        return {
            "piqe": piqe,
            "brisque": brisque,
            "niqe": niqe,
        }

    iqa_rows = []
    iqa_errors = []

    for idx, row in enumerate(manifest.itertuples(index=False), start=1):
        try:
            scores = extract_iqa_scores(row.path)
            iqa_rows.append({"image_id": row.image_id, **scores})
        except Exception as exc:
            iqa_errors.append({
                "image_id": row.image_id,
                "path": row.path,
                "error": repr(exc),
            })

        if idx % 100 == 0:
            print(f"Processed IQA {idx}/{len(manifest)} images")

    if iqa_errors:
        err_df = pd.DataFrame(iqa_errors)
        err_out = IQA_OUT.parent / "features_iqa_errors_biggan_midjourney_multiclass_train.csv"
        err_df.to_csv(err_out, index=False)
        print("IQA errors:", len(iqa_errors))
        print("Saved IQA error log:", err_out.resolve())
        display(err_df.head())

    if not iqa_rows:
        raise RuntimeError("No IQA features were extracted. Check the IQA error log.")

    iqa_df = pd.DataFrame(iqa_rows)
    iqa_df.to_csv(IQA_OUT, index=False)
    print("Saved IQA features:", IQA_OUT.resolve())

display(iqa_df.head())


## 4. Bangun feature vector

In [ ]:
if FFT_OUT.exists():
    fft_df = pd.read_csv(FFT_OUT)
    print("Loaded FFT features:", FFT_OUT.resolve())
else:
    def load_gray(path):
        return np.asarray(Image.open(path).convert("L"), dtype=np.float32)

    def extract_fft_mean(gray):
        f = np.fft.fft2(gray)
        mag = np.abs(f)
        phase = np.angle(f)
        return {
            "fft_mag_mean": float(np.mean(mag)),
            "fft_phase_mean": float(np.mean(phase)),
            "fft_phase_cos_mean": float(np.mean(np.cos(phase))),
            "fft_phase_sin_mean": float(np.mean(np.sin(phase))),
        }

    fft_rows = []
    for idx, row in enumerate(manifest.itertuples(index=False), start=1):
        gray = load_gray(row.path)
        feats = extract_fft_mean(gray)
        fft_rows.append({"image_id": row.image_id, **feats})
        if idx % 200 == 0:
            print(f"Processed FFT {idx}/{len(manifest)} images")

    fft_df = pd.DataFrame(fft_rows)
    fft_df.to_csv(FFT_OUT, index=False)
    print("Saved FFT features:", FFT_OUT.resolve())

print("FFT shape:", fft_df.shape)
display(fft_df.head())


In [ ]:
data = manifest.merge(fft_df, on="image_id", how="inner").merge(iqa_df, on="image_id", how="inner")
meta_cols = ["image_id", "path", "relative_path", "generator", "subset_name", "split", "class_name", "content_id", "content_label", "is_real", "y_ai"]
feature_cols = [
    c for c in data.columns
    if c.startswith(("fft_", "clip_feat_")) or c in ["piqe", "brisque", "niqe"]
]

data.to_csv(VECTOR_OUT, index=False)
print("Merged shape:", data.shape)
print("Feature dims:", len(feature_cols))
print("Saved vector:", VECTOR_OUT.resolve())
display(data.head())

In [ ]:
plot_cols = ["fft_mag_mean", "fft_phase_mean", "fft_phase_cos_mean", "fft_phase_sin_mean", "piqe", "brisque", "niqe"]
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()
for ax, col in zip(axes, plot_cols):
    for label, color in [(1, "tab:red"), (0, "tab:blue")]:
        ax.hist(data[data["y_ai"] == label][col], bins=25, alpha=0.6, label=f"y_ai={label}", color=color)
    ax.set_title(col)
    ax.legend()
for ax in axes[len(plot_cols):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Split 80:20 dengan stratifikasi gabungan

In [ ]:
X = data[feature_cols].to_numpy(dtype=np.float32)
y = data["y_ai"].to_numpy(dtype=np.int64)
stratify_key = data["generator"].astype(str) + "|" + data["y_ai"].astype(str) + "|" + data["content_label"].astype(str)

X_train, X_eval, y_train, y_eval, train_idx, eval_idx = train_test_split(
    X, y, data.index.to_numpy(), test_size=0.2, random_state=42, stratify=stratify_key
)

print("Train:", X_train.shape, y_train.shape)
print("Eval:", X_eval.shape, y_eval.shape)

## 6. Train baseline

In [ ]:
MLP_HIDDEN = (64, 32)
models = {
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
    ),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(hidden_layer_sizes=MLP_HIDDEN, activation="relu", max_iter=500, random_state=42)),
    ]),
}

results = []
pred_store = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    score_ai = model.predict_proba(X_eval)[:, 1]
    pred_ai = (score_ai >= 0.5).astype(np.int64)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_eval, pred_ai),
        "f1": f1_score(y_eval, pred_ai),
        "auroc": roc_auc_score(y_eval, score_ai),
    })
    pred_store[name] = {"score_ai": score_ai, "pred_ai": pred_ai}

results_df = pd.DataFrame(results).sort_values(by=["auroc", "f1", "accuracy"], ascending=False).reset_index(drop=True)
display(results_df)

## 7. Confusion matrix model terbaik

In [ ]:
best_model = results_df.iloc[0]["model"]
print("Best model:", best_model)
print()
print(classification_report(y_eval, pred_store[best_model]["pred_ai"], target_names=["nature", "ai"]))
cm = confusion_matrix(y_eval, pred_store[best_model]["pred_ai"])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["nature", "ai"]).plot(cmap="Blues")
plt.show()

## 8. Simpan hasil

In [ ]:
metrics_df = results_df.copy()
metrics_df["feature_set"] = "iqa_plus_fft"
metrics_df["iqa_metrics"] = "piqe,brisque,niqe"
metrics_df["split_protocol"] = "train_holdout_80_20_stratified_generator_label_class"
metrics_df["subset_name"] = SUBSET_NAME
metrics_df["n_total"] = len(data)
metrics_df["n_train"] = len(X_train)
metrics_df["n_eval"] = len(X_eval)
metrics_df.to_csv(METRICS_OUT, index=False)

base_eval = data.iloc[eval_idx][["image_id", "path", "generator", "class_name", "content_id", "content_label", "y_ai"]].copy()
pred_parts = []
for name, pred in pred_store.items():
    part = base_eval.copy()
    part["model"] = name
    part["score_ai"] = pred["score_ai"]
    part["pred_ai"] = pred["pred_ai"]
    pred_parts.append(part)

pred_df = pd.concat(pred_parts, ignore_index=True)
pred_df.to_csv(PRED_OUT, index=False)

print("Saved metrics:", METRICS_OUT.resolve())
print("Saved predictions:", PRED_OUT.resolve())
display(metrics_df)
display(pred_df.head())

## 9. Catatan interpretasi

Notebook ini menguji apakah kombinasi fitur spektral sederhana dan metrik NR-IQA tradisional memberi sinyal yang lebih kuat dibanding masing-masing keluarga fitur secara terpisah.